[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aims-foundations/torch_measure/blob/main/tutorials/quickstart.ipynb)

# torch_measure: Quick Start Tutorial

**torch_measure** is a PyTorch-native toolkit for measurement science. It provides:

- **Item Response Theory (IRT)** models: Rasch, 2PL, 3PL, Amortized IRT
- **Computerized Adaptive Testing (CAT)** with Fisher information-based item selection
- **Psychometric metrics**: reliability, calibration, scalability, DIF detection
- **Factor models**: logistic factor model, bifactor, Varimax/Promax rotation
- **Data utilities**: response matrices, masking strategies, transforms

This tutorial walks through the main features with synthetic data.

## 1. Setup

In [ ]:
try:
    import google.colab
    !git clone https://github.com/aims-foundations/torch_measure.git
    !pip install -e "torch_measure[data]"
except ImportError:
    pass  # Already installed locally

import torch
import matplotlib.pyplot as plt

from torch_measure.models import Rasch, TwoPL, ThreePL, AmortizedIRT, LogisticFM, varimax_rotation
from torch_measure.cat import AdaptiveTester
from torch_measure.data import ResponseMatrix, random_mask, l_mask, row_mask, col_mask
from torch_measure.metrics import (
    cronbach_alpha, infit_statistics, outfit_statistics,
    item_total_correlation, tetrachoric_correlation,
    mokken_scalability, expected_calibration_error, brier_score,
    ability_standard_errors, difficulty_standard_errors, discrimination_standard_errors,
)

plt.rcParams["figure.dpi"] = 100
print(f"torch_measure imported successfully")

## 2. Generating Synthetic Data

We generate a response matrix from known IRT parameters so we can later validate that our model recovers the true values.

The Rasch model: $P(\text{correct}) = \sigma(\theta_n - b_j)$ where $\theta$ is ability and $b$ is difficulty.

In [ ]:
torch.manual_seed(42)

n_subjects, n_items = 200, 100

# True parameters
true_ability = torch.randn(n_subjects)  # N(0,1) abilities
true_difficulty = torch.randn(n_items)  # N(0,1) difficulties

# Generate responses from the Rasch model
logits = true_ability.unsqueeze(1) - true_difficulty.unsqueeze(0)  # (N, M)
true_probs = torch.sigmoid(logits)
responses = torch.bernoulli(true_probs)

print(f"Response matrix shape: {responses.shape}")
print(f"Overall accuracy: {responses.mean():.3f}")

In [ ]:
# Wrap in ResponseMatrix for convenient properties
rm = ResponseMatrix(responses)
print(rm)
print(f"Density (fraction observed): {rm.density:.2%}")
print(f"Subject mean scores: min={rm.subject_means.min():.2f}, max={rm.subject_means.max():.2f}")
print(f"Item facility (easiness): min={rm.item_means.min():.2f}, max={rm.item_means.max():.2f}")

In [ ]:
# Visualize the response matrix (sorted by ability and difficulty)
sorted_subj = true_ability.argsort()
sorted_item = true_difficulty.argsort()
sorted_responses = responses[sorted_subj][:, sorted_item]

fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(sorted_responses.numpy(), aspect="auto", cmap="RdYlGn", interpolation="nearest")
ax.set_xlabel("Items (sorted by difficulty)")
ax.set_ylabel("Subjects (sorted by ability)")
ax.set_title("Response Matrix (green=correct, red=incorrect)")
plt.tight_layout()
plt.show()

The Guttman pattern is visible: high-ability subjects (top) answer most items correctly, easy items (left) are answered correctly by most subjects.

## 3. Fitting IRT Models

torch_measure provides three standard IRT models with increasing complexity:
- **Rasch (1PL)**: $P = \sigma(\theta - b)$
- **2PL**: $P = \sigma(a(\theta - b))$ — adds item discrimination $a$
- **3PL**: $P = c + (1-c)\sigma(a(\theta - b))$ — adds guessing $c$

In [ ]:
# Fit all three models
rasch = Rasch(n_subjects=n_subjects, n_items=n_items)
twopl = TwoPL(n_subjects=n_subjects, n_items=n_items)
threepl = ThreePL(n_subjects=n_subjects, n_items=n_items)

history_rasch = rasch.fit(responses, max_epochs=300, verbose=False)
history_twopl = twopl.fit(responses, max_epochs=300, verbose=False)
history_threepl = threepl.fit(responses, max_epochs=300, verbose=False)

print(f"Final loss — Rasch: {history_rasch['losses'][-1]:.4f}, "
      f"2PL: {history_twopl['losses'][-1]:.4f}, "
      f"3PL: {history_threepl['losses'][-1]:.4f}")

In [ ]:
# Plot training loss curves
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_rasch["losses"], label="Rasch (1PL)")
ax.plot(history_twopl["losses"], label="2PL")
ax.plot(history_threepl["losses"], label="3PL")
ax.set_xlabel("Epoch")
ax.set_ylabel("Negative Log-Likelihood")
ax.set_title("Training Loss Curves")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Since data was generated from Rasch, check parameter recovery
# IRT parameters are identifiable only up to a linear transform, so we correlate
est_ability = rasch.ability.detach()
est_difficulty = rasch.difficulty.detach()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(true_ability.numpy(), est_ability.numpy(), alpha=0.5, s=15)
axes[0].set_xlabel("True Ability")
axes[0].set_ylabel("Estimated Ability")
axes[0].set_title(f"Ability Recovery (r={torch.corrcoef(torch.stack([true_ability, est_ability]))[0,1]:.3f})")
axes[0].plot([-3, 3], [-3, 3], "k--", alpha=0.3)
axes[0].grid(True, alpha=0.3)

axes[1].scatter(true_difficulty.numpy(), est_difficulty.numpy(), alpha=0.5, s=15)
axes[1].set_xlabel("True Difficulty")
axes[1].set_ylabel("Estimated Difficulty")
axes[1].set_title(f"Difficulty Recovery (r={torch.corrcoef(torch.stack([true_difficulty, est_difficulty]))[0,1]:.3f})")
axes[1].plot([-3, 3], [-3, 3], "k--", alpha=0.3)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 2PL-specific: inspect discrimination parameters
disc = twopl.discrimination.detach()
print(f"Discrimination — mean: {disc.mean():.2f}, std: {disc.std():.2f}")
print(f"Range: [{disc.min():.2f}, {disc.max():.2f}]")

# 3PL-specific: inspect guessing parameters
guess = threepl.guessing.detach()
print(f"\nGuessing — mean: {guess.mean():.3f}, std: {guess.std():.3f}")
print(f"Range: [{guess.min():.3f}, {guess.max():.3f}]")
print("(Should be near 0 since data was generated without guessing)")

## 4. Train/Test Evaluation

Use masking to hold out 20% of responses, fit on the rest, and evaluate generalization.

In [ ]:
torch.manual_seed(42)

observed = torch.ones_like(responses, dtype=torch.bool)
train_mask, test_mask = random_mask(observed, train_frac=0.8)

print(f"Training entries: {train_mask.sum().item():,}")
print(f"Test entries: {test_mask.sum().item():,}")

# Fit on training data only
model_eval = Rasch(n_subjects=n_subjects, n_items=n_items)
history = model_eval.fit(responses, mask=train_mask, max_epochs=300, verbose=False)

# Evaluate on test set
probs = model_eval.predict().detach()
ece = expected_calibration_error(probs, responses, mask=test_mask)
bs = brier_score(probs, responses, mask=test_mask)

print(f"\nTest set metrics:")
print(f"  Expected Calibration Error: {ece:.4f}")
print(f"  Brier Score: {bs:.4f}")

## 5. Psychometric Metrics

torch_measure provides standard psychometric quality measures.

In [ ]:
# Internal consistency
alpha = cronbach_alpha(responses)
print(f"Cronbach's alpha: {alpha:.3f}")

# Mokken scalability
scalability = mokken_scalability(responses)
print(f"Mokken H (overall): {scalability['H']:.3f}")
print(f"  H >= 0.5 = strong scale, 0.4-0.5 = medium, 0.3-0.4 = weak")

In [ ]:
# Item fit statistics using the Rasch model predictions
probs_rasch = rasch.predict().detach()
infit = infit_statistics(probs_rasch, responses)
outfit = outfit_statistics(probs_rasch, responses)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(n_items), infit.numpy(), alpha=0.7)
axes[0].axhline(y=1.0, color="k", linestyle="--", alpha=0.5)
axes[0].axhline(y=1.3, color="r", linestyle="--", alpha=0.5, label="Underfit threshold")
axes[0].axhline(y=0.7, color="b", linestyle="--", alpha=0.5, label="Overfit threshold")
axes[0].set_xlabel("Item")
axes[0].set_ylabel("Infit MNSQ")
axes[0].set_title("Infit Statistics")
axes[0].legend(fontsize=8)

axes[1].bar(range(n_items), outfit.numpy(), alpha=0.7)
axes[1].axhline(y=1.0, color="k", linestyle="--", alpha=0.5)
axes[1].axhline(y=1.3, color="r", linestyle="--", alpha=0.5, label="Underfit threshold")
axes[1].axhline(y=0.7, color="b", linestyle="--", alpha=0.5, label="Overfit threshold")
axes[1].set_xlabel("Item")
axes[1].set_ylabel("Outfit MNSQ")
axes[1].set_title("Outfit Statistics")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

n_underfit = (infit > 1.3).sum().item()
n_overfit = (infit < 0.7).sum().item()
print(f"Items with poor infit: {n_underfit} underfit, {n_overfit} overfit out of {n_items}")

In [ ]:
# Item-total correlation
itc = item_total_correlation(responses)

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(n_items), itc.numpy(), alpha=0.7)
ax.axhline(y=0.2, color="r", linestyle="--", alpha=0.5, label="Minimum threshold")
ax.set_xlabel("Item")
ax.set_ylabel("Corrected Item-Total Correlation")
ax.set_title("Item Discrimination via Item-Total Correlation")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

weak_items = (itc < 0.2).nonzero(as_tuple=True)[0]
print(f"Weak items (r_it < 0.2): {weak_items.tolist() if len(weak_items) > 0 else 'None'}")

In [ ]:
# Tetrachoric correlation heatmap
corr = tetrachoric_correlation(responses)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr.numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xlabel("Item")
ax.set_ylabel("Item")
ax.set_title("Tetrachoric Correlation Matrix")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## 6. Uncertainty Quantification

torch_measure provides two complementary approaches for quantifying uncertainty in IRT parameter estimates:

- **Fisher-based standard errors** (frequentist): `SE = 1 / sqrt(Fisher information)`. Fast, works with any fitted model.
- **SVI posteriors** (Bayesian): Full variational posterior from Stochastic Variational Inference. Richer but requires refitting with `method="svi"`.

### 6a. Fisher-based Standard Errors

These work with any fitted model — no refitting needed. The standard error for ability is $\text{SE}(\theta_i) = 1/\sqrt{\sum_j I_j(\theta_i)}$, where $I_j$ is the Fisher information of item $j$.

In [ ]:
# Ability standard errors from the fitted Rasch model
ability_se = ability_standard_errors(est_ability, est_difficulty)
difficulty_se = difficulty_standard_errors(est_ability, est_difficulty, responses)

print(f"Ability SE — mean: {ability_se.mean():.3f}, range: [{ability_se.min():.3f}, {ability_se.max():.3f}]")
print(f"Difficulty SE — mean: {difficulty_se.mean():.3f}, range: [{difficulty_se.min():.3f}, {difficulty_se.max():.3f}]")

# For the 2PL model, we can also get discrimination SE
disc_se = discrimination_standard_errors(
    twopl.ability.detach(), twopl.difficulty.detach(),
    twopl.discrimination.detach(), responses,
)
print(f"\n2PL Discrimination SE — mean: {disc_se.mean():.3f}, range: [{disc_se.min():.3f}, {disc_se.max():.3f}]")

In [ ]:
# Visualize: ability estimates with 95% confidence intervals and SE vs ability
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: sorted ability estimates with error bands
sorted_idx = est_ability.argsort()
sorted_ability = est_ability[sorted_idx]
sorted_se = ability_se[sorted_idx]
x = range(len(sorted_ability))
axes[0].fill_between(x, (sorted_ability - 1.96 * sorted_se).numpy(),
                     (sorted_ability + 1.96 * sorted_se).numpy(), alpha=0.3, label="95% CI")
axes[0].plot(x, sorted_ability.numpy(), "k-", linewidth=0.5)
axes[0].set_xlabel("Subject (sorted by ability)")
axes[0].set_ylabel("Ability")
axes[0].set_title("Ability Estimates with 95% Confidence Intervals")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: SE as a function of ability (U-shaped for Rasch)
axes[1].scatter(est_ability.numpy(), ability_se.numpy(), alpha=0.5, s=15)
axes[1].set_xlabel("Estimated Ability")
axes[1].set_ylabel("Standard Error")
axes[1].set_title("Standard Error vs. Ability (expected U-shape)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Extreme abilities have larger SE — less information at the tails.")

### 6b. SVI Posteriors (Bayesian Uncertainty)

When fitting with `method="svi"`, the returned history includes a `"posterior"` dict with variational `loc` and `scale` for each parameter. This gives full distributional uncertainty, not just a point estimate.

In [ ]:
# Fit the same Rasch model with SVI to get Bayesian posteriors
rasch_svi = Rasch(n_subjects=n_subjects, n_items=n_items)
history_svi = rasch_svi.fit(responses, method="svi", max_epochs=500, verbose=False)

posterior = history_svi["posterior"]
print(f"Posterior keys: {list(posterior.keys())}")
print(f"Ability  — loc shape: {posterior['ability']['loc'].shape}, "
      f"scale mean: {posterior['ability']['scale'].mean():.3f}")
print(f"Difficulty — loc shape: {posterior['difficulty']['loc'].shape}, "
      f"scale mean: {posterior['difficulty']['scale'].mean():.3f}")

# Compare Fisher SE vs SVI posterior scale
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(ability_se.numpy(), posterior["ability"]["scale"].numpy(), alpha=0.5, s=15)
axes[0].set_xlabel("Fisher SE")
axes[0].set_ylabel("SVI Posterior Scale")
axes[0].set_title("Ability: Fisher SE vs. SVI Scale")
lim = max(ability_se.max().item(), posterior["ability"]["scale"].max().item()) * 1.1
axes[0].plot([0, lim], [0, lim], "k--", alpha=0.3)
axes[0].grid(True, alpha=0.3)

axes[1].scatter(difficulty_se.numpy(), posterior["difficulty"]["scale"].numpy(), alpha=0.5, s=15)
axes[1].set_xlabel("Fisher SE")
axes[1].set_ylabel("SVI Posterior Scale")
axes[1].set_title("Difficulty: Fisher SE vs. SVI Scale")
lim = max(difficulty_se.max().item(), posterior["difficulty"]["scale"].max().item()) * 1.1
axes[1].plot([0, lim], [0, lim], "k--", alpha=0.3)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Both methods agree on relative uncertainty — Fisher SE is fast, SVI gives full posteriors.")

## 7. Computerized Adaptive Testing (CAT)

CAT selects the most informative items for each subject, estimating ability with fewer items than a full test.

In [ ]:
# Use the fitted Rasch model as the item bank
# Simulate a new subject with known ability
torch.manual_seed(123)
true_subject_ability = 1.5  # above average
subject_logits = true_subject_ability - rasch.difficulty.detach()
subject_responses = torch.bernoulli(torch.sigmoid(subject_logits))

# Run CAT with Fisher information strategy
tester_fisher = AdaptiveTester(rasch, strategy="fisher")
result_fisher = tester_fisher.run(subject_responses, budget=20, lr=0.3, n_steps=30)

# Run CAT with random strategy (baseline)
torch.manual_seed(123)
tester_random = AdaptiveTester(rasch, strategy="random")
result_random = tester_random.run(subject_responses, budget=20, lr=0.3, n_steps=30)

print(f"True ability: {true_subject_ability:.2f}")
print(f"Fisher estimate (20 items): {result_fisher['ability']:.2f}")
print(f"Random estimate (20 items): {result_random['ability']:.2f}")

In [ ]:
# Plot ability trajectory
fig, ax = plt.subplots(figsize=(8, 4))
items_range = range(1, 21)
ax.plot(items_range, result_fisher["ability_trajectory"], "o-", label="Fisher (max info)", markersize=4)
ax.plot(items_range, result_random["ability_trajectory"], "s-", label="Random", markersize=4, alpha=0.7)
ax.axhline(y=true_subject_ability, color="k", linestyle="--", alpha=0.5, label=f"True ability ({true_subject_ability})")
ax.set_xlabel("Number of Items Administered")
ax.set_ylabel("Estimated Ability")
ax.set_title("Adaptive Testing: Ability Estimation Trajectory")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Fisher information-based selection converges to the true ability faster and with less variability than random selection.

## 8. Multidimensional Factor Model

When items measure multiple latent traits, a factor model captures the structure.

In [ ]:
# Generate data with 3 latent factors
torch.manual_seed(42)
n_s, n_i, k = 150, 60, 3

# True factor loadings: items 0-19 load on factor 1, 20-39 on factor 2, 40-59 on factor 3
true_U = torch.randn(n_s, k)  # Subject abilities
true_V = torch.zeros(n_i, k)  # Item loadings
true_V[:20, 0] = torch.randn(20) * 1.5
true_V[20:40, 1] = torch.randn(20) * 1.5
true_V[40:, 2] = torch.randn(20) * 1.5
true_Z = torch.randn(n_i) * 0.5  # Intercepts

factor_logits = true_U @ true_V.T + true_Z.unsqueeze(0)
factor_responses = torch.bernoulli(torch.sigmoid(factor_logits))

# Fit LogisticFM
fm = LogisticFM(n_subjects=n_s, n_items=n_i, n_factors=k)
history_fm = fm.fit(factor_responses, max_epochs=300, verbose=False)
print(f"Final loss: {history_fm['losses'][-1]:.4f}")

In [ ]:
# Apply Varimax rotation for interpretable loadings
raw_loadings = fm.loadings  # (n_items, K)
rotated_loadings, rotation_matrix = varimax_rotation(raw_loadings)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(raw_loadings.numpy(), aspect="auto", cmap="RdBu_r", vmin=-2, vmax=2)
axes[0].set_xlabel("Factor")
axes[0].set_ylabel("Item")
axes[0].set_title("Raw Loadings")
plt.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(rotated_loadings.numpy(), aspect="auto", cmap="RdBu_r", vmin=-2, vmax=2)
axes[1].set_xlabel("Factor")
axes[1].set_ylabel("Item")
axes[1].set_title("Varimax-Rotated Loadings")
plt.colorbar(im1, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.show()
print("Rotated loadings should show clearer block-diagonal structure.")

## 9. Masking Strategies

torch_measure provides several masking strategies for train/test splitting, each testing different generalization capabilities.

In [ ]:
torch.manual_seed(42)
small_obs = torch.ones(20, 30, dtype=torch.bool)

masks = {
    "Random": random_mask(small_obs, train_frac=0.7),
    "L-shaped": l_mask(small_obs, row_frac=0.7, col_frac=0.7),
    "Row-based": row_mask(small_obs, train_frac=0.7, exposure_rate=0.2),
    "Column-based": col_mask(small_obs, train_frac=0.7, exposure_rate=0.2),
}

fig, axes = plt.subplots(2, 4, figsize=(14, 5))
for idx, (name, (train_m, test_m)) in enumerate(masks.items()):
    # Train mask
    axes[0, idx].imshow(train_m.float().numpy(), aspect="auto", cmap="Blues", vmin=0, vmax=1)
    axes[0, idx].set_title(f"{name}\n(train: {train_m.sum().item()})")
    axes[0, idx].set_ylabel("Train" if idx == 0 else "")
    # Test mask
    axes[1, idx].imshow(test_m.float().numpy(), aspect="auto", cmap="Oranges", vmin=0, vmax=1)
    axes[1, idx].set_ylabel("Test" if idx == 0 else "")
    axes[1, idx].set_title(f"(test: {test_m.sum().item()})")

plt.suptitle("Masking Strategies", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

- **Random**: Standard hold-out. Tests interpolation.
- **L-shaped**: Hold out the bottom-right block. Tests generalization to new subjects on new items.
- **Row-based**: Partially observe held-out subjects. Tests cold-start for subjects.
- **Column-based**: Partially observe held-out items. Tests cold-start for items.

## 10. Handling Missing Data

Real response matrices often have missing entries. torch_measure handles NaN values automatically.

In [ ]:
torch.manual_seed(42)

# Create response matrix with 15% missing data
responses_missing = responses.clone()
missing_mask = torch.rand_like(responses.float()) < 0.15
responses_missing[missing_mask] = float("nan")

rm_missing = ResponseMatrix(responses_missing)
print(f"Density: {rm_missing.density:.2%} (vs 100% for complete data)")

# Fit on incomplete data
model_missing = Rasch(n_subjects=n_subjects, n_items=n_items)
history_missing = model_missing.fit(responses_missing, max_epochs=300, verbose=False)

# Compare parameter estimates
est_full = rasch.ability.detach()
est_missing = model_missing.ability.detach()

r = torch.corrcoef(torch.stack([est_full, est_missing]))[0, 1]
print(f"Correlation between full vs missing-data ability estimates: {r:.3f}")
print("High correlation indicates robustness to missing data.")

## 11. Advanced: Amortized IRT

AmortizedIRT predicts item parameters from embeddings (e.g., BERT representations of test questions). This enables **zero-shot prediction** on new items without refitting.

In [ ]:
torch.manual_seed(42)
n_s_amort, n_i_amort, emb_dim = 50, 30, 64

# Simulate item embeddings and responses
embeddings = torch.randn(n_i_amort, emb_dim)
responses_amort = torch.bernoulli(torch.full((n_s_amort, n_i_amort), 0.5))

# Fit amortized model (learns embedding -> item params mapping)
amort = AmortizedIRT(
    n_subjects=n_s_amort,
    n_items=n_i_amort,
    embedding_dim=emb_dim,
    hidden_dim=64,
    n_layers=2,
    pl=2,  # 2PL model
)
history_amort = amort.fit(responses_amort, embeddings, max_epochs=100, verbose=False)

print(f"Final loss: {history_amort['losses'][-1]:.4f}")
print(f"Learned difficulties: {amort.difficulty[:5]}")
print(f"Learned discriminations: {amort.discrimination[:5]}")

In [ ]:
# Zero-shot: predict parameters for a new item from its embedding
# (In practice, this would be a new test question's BERT embedding)
new_embedding = torch.randn(1, emb_dim)

# Temporarily expand the model to include the new item
all_embeddings = torch.cat([embeddings, new_embedding], dim=0)
amort_extended = AmortizedIRT(
    n_subjects=n_s_amort, n_items=n_i_amort + 1,
    embedding_dim=emb_dim, hidden_dim=64, n_layers=2, pl=2,
)
# Copy the learned network weights
amort_extended.item_net.load_state_dict(amort.item_net.state_dict())
amort_extended.set_embeddings(all_embeddings)

print(f"Predicted difficulty for new item: {amort_extended.difficulty[-1]:.3f}")
print(f"Predicted discrimination for new item: {amort_extended.discrimination[-1]:.3f}")
print("\nNo refitting needed — parameters predicted directly from embedding!")

## 12. Summary

This tutorial covered:

| Feature | Module | Key API |
|---------|--------|---------|
| IRT Models | `torch_measure.models` | `Rasch`, `TwoPL`, `ThreePL`, `AmortizedIRT` |
| Factor Models | `torch_measure.models` | `LogisticFM`, `varimax_rotation` |
| Model Fitting | `torch_measure.fitting` | `.fit(method="mle"/"em"/"jml"/"svi")` |
| Uncertainty (Fisher) | `torch_measure.metrics` | `ability_standard_errors`, `difficulty_standard_errors` |
| Uncertainty (Bayesian) | `torch_measure.fitting` | `history["posterior"]` from SVI fitting |
| Adaptive Testing | `torch_measure.cat` | `AdaptiveTester(strategy="fisher")` |
| Psychometric Metrics | `torch_measure.metrics` | `cronbach_alpha`, `infit_statistics`, `mokken_scalability` |
| Data Utilities | `torch_measure.data` | `ResponseMatrix`, `random_mask`, `l_mask` |

### Next Steps

- **API Reference**: See the full docs at [torch-measure.readthedocs.io](https://torch-measure.readthedocs.io)
- **GitHub**: [github.com/aims-foundations/torch_measure](https://github.com/aims-foundations/torch_measure)
- **Install**: `pip install torch_measure`